In [1]:
import pandas as pd
import numpy as np
import scanpy as sc
import os
from datetime import datetime
import matplotlib.pyplot as plt

os.environ['TZ'] = 'Europe/Berlin'
import time
time.tzset()

def checkpoint(msg):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}", flush=True)


In [2]:
BASE = "/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune"
H5AD = f"{BASE}/data/03_HIHAtlas/HIHA_hgnc.h5ad"
NET_LEVEL1_CSV = f"{BASE}/data/02_TICA/TICA_gene_markers_level1_table_s2.csv"
NET_LEVEL2_CSV = f"{BASE}/data/02_TICA/TICA_gene_markers_level2_table_s3.csv"
OUT_DIR = f"{BASE}/results/09_HIHAtlas"
ANNOTATED_H5AD_OUT = f"{BASE}/data/03_HIHAtlas/immune_cells_HIHA_TICAannot_gpu.h5ad"
os.makedirs(OUT_DIR, exist_ok=True)
def load_net(path):
    df = pd.read_csv(path)
    net = df.rename(columns={"cell_type": "source", "gene": "target"})[["source", "target"]].copy()
    net["weight"] = 1.0  # unweighted — all marker genes contribute equally
    
    return net



adata = sc.read_h5ad(H5AD)
net_level1 = load_net(NET_LEVEL1_CSV)
net_level2 = load_net(NET_LEVEL2_CSV)


In [ ]:
print(adata.n_vars)

genes_net = set(net_level1["target"].str.upper())

print(net_level1.head())


32357
    source target  weight
0  B cells  MS4A1     1.0
1  B cells  CD79A     1.0
2  B cells   CD83     1.0
3  B cells  CD79B     1.0
4  B cells   CD37     1.0


In [27]:
print(adata.var.columns)
print("NaN in gene_symbols:", adata.var['gene_symbols'].isna().sum())
print(adata.var_names[:10].tolist())

print("NaN in feature_name:", adata.var['feature_name'].isna().sum())
print(adata.var['feature_name'][:10].tolist())

Index(['mito', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts',
       'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts',
       'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean',
       'std', 'gene_symbols', 'feature_is_filtered', 'feature_name',
       'feature_reference', 'feature_biotype', 'feature_length',
       'feature_type'],
      dtype='object')
NaN in gene_symbols: 31132
['ENSG00000243485', 'ENSG00000237613', 'ENSG00000186092', 'ENSG00000239945', 'ENSG00000239906', 'ENSG00000241599', 'ENSG00000236601', 'ENSG00000284733', 'ENSG00000235146', 'ENSG00000284662']
NaN in feature_name: 0
['MIR1302-2HG', 'FAM138A', 'OR4F5', 'ENSG00000239945', 'ENSG00000239906', 'ENSG00000241599', 'ENSG00000236601', 'OR4F29', 'ENSG00000235146', 'OR4F16']


In [4]:
print(adata.obs.columns.tolist())

# nach möglichen Spalten suchen, die Tumor/Healthy unterscheiden
for col in adata.obs.columns:
    if adata.obs[col].dtype.name in ["category", "object"]:
        unique_vals = adata.obs[col].unique()
        if len(unique_vals) < 50:  # nur überschaubare kategoriale Spalten anzeigen
            print(f"\n{col}: {unique_vals}")

['cohort.cohortGuid', 'sample.sampleKitGuid', 'specimen.specimenGuid', 'pipeline.fileGuid', 'subject.birthYear', 'subject.ageAtFirstDraw', 'subject.ageGroup', 'subject.race', 'subject.ethnicity', 'subject.cmv', 'subject.bmi', 'sample.visitName', 'sample.drawYear', 'sample.subjectAgeAtDraw', 'batch_id', 'pool_id', 'chip_id', 'well_id', 'original_barcodes', 'cell_name', 'n_reads', 'n_umis', 'n_genes', 'total_counts_mito', 'pct_counts_mito', 'doublet_score', 'AIFI_L1', 'AIFI_L2', 'AIFI_L3', 'disease_ontology_term_id', 'tissue_ontology_term_id', 'tissue_type', 'suspension_type', 'assay_ontology_term_id', 'sex_ontology_term_id', 'donor_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'cell_type_ontology_term_id', 'is_primary_data', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid']

cohort.cohortGuid: ['BR1', 'BR2', 'UP1']
Categories (3, object): ['BR1', 'BR2', 'UP1']

subject.ageGroup

In [34]:
n_cells_per_sample = adata.obs['donor_id'].value_counts()
print(n_cells_per_sample.sort_values())

donor_id
UP1024     6632
BR2035    10217
BR2009    11724
BR2053    12571
BR2029    12658
          ...  
BR2030    22728
BR1051    22931
BR2010    23108
BR2025    24539
BR2048    26131
Name: count, Length: 108, dtype: int64


In [4]:
print(adata.obs['cell_type'].values
      )

['mucosal-associated invariant T cell', 'naive thymus-derived CD4-positive, alpha-beta..., 'CD14-positive, CD16-negative classical monocyte', 'central memory CD4-positive, alpha-beta T cell', 'CD14-positive, CD16-negative classical monocyte', ..., 'naive thymus-derived CD8-positive, alpha-beta..., 'naive thymus-derived CD4-positive, alpha-beta..., 'naive regulatory T cell', 'memory B cell', 'central memory CD4-positive, alpha-beta T cell']
Length: 1821725
Categories (38, object): ['common myeloid progenitor', 'common lymphoid progenitor', 'T cell', 'erythrocyte', ..., 'naive regulatory T cell', 'memory regulatory T cell', 'CD8aa(I) thymocyte', 'Axl+ dendritic cell, human']


In [10]:
print(adata.var.columns.tolist())
genes_adata = set(adata.var_names.str.upper())
print(f"Anzahl Gene in adata: {len(genes_adata)}")

# nach möglichen Spalten suchen, die Tumor/Healthy unterscheiden
for col in adata.var.columns:
    if adata.var[col].dtype.name in ["category", "object"]:
        unique_vals = adata.var[col].unique()
        if len(unique_vals) < 50:  # nur überschaubare kategoriale Spalten anzeigen
            print(f"\n{col}: {unique_vals}")

['mito', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std', 'gene_symbols', 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type']
Anzahl Gene in adata: 32357

feature_reference: ['NCBITaxon:9606']
Categories (1, object): ['NCBITaxon:9606']

feature_biotype: ['gene']
Categories (1, object): ['gene']

feature_type: ['lncRNA', 'protein_coding', 'transcribed_unprocessed_pseudogene', 'IG_V_pseudogene', 'unprocessed_pseudogene', ..., 'IG_C_pseudogene', 'TR_J_pseudogene', 'IG_J_pseudogene', 'IG_D_gene', 'artifact']
Length: 21
Categories (21, object): ['IG_C_gene', 'IG_C_pseudogene', 'IG_D_gene', 'IG_J_gene', ..., 'transcribed_processed_pseudogene', 'transcribed_unitary_pseudogene', 'transcribed_unprocessed_pseudogene', 'unprocessed_pseudogene']


In [26]:
# Genes in adata
genes_adata = set(adata.var['feature_name'].str.upper())  # or .str.lower(), depending on your net

# Genes in network
genes_net = set(net_level1["target"].str.upper())  # adjust column name if needed

overlap = genes_adata & genes_net
print(f"Genes in adata: {len(genes_adata)}")
print(f"Genes in net:   {len(genes_net)}")
print(f"Overlap:        {len(overlap)}")

Genes in adata: 32345
Genes in net:   150
Overlap:        147


### Convert CellxGene h5ad: map ENSEMBL IDs in feature_name to HGNC symbols, set var_names = feature_name, remove duplicates, save as new h5ad.

In [30]:
#!/usr/bin/env python3


import scanpy as sc
import mygene
import pandas as pd
import numpy as np

INPUT = "/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/data/03_HIHAtlas/HumanImmuneHealthAtlas_all.h5ad"
OUTPUT = "/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/data/03_HIHAtlas/HIHA_hgnc.h5ad"

print(f"Loading {INPUT}...")
adata = sc.read_h5ad(INPUT)
print(f"Original: {adata.n_obs:,} cells, {adata.n_vars:,} genes")

# feature_name als Basis
print("Using feature_name as gene symbols...")

# ENSEMBL-IDs erkennen und mappen
ens_mask = adata.var['feature_name'].str.startswith('ENSG')
if ens_mask.any():
    print(f"Mapping {ens_mask.sum()} ENSEMBL-IDs to HGNC symbols...")
    
    mg = mygene.MyGeneInfo()
    ens_ids = adata.var.loc[ens_mask, 'feature_name'].str.split('.').str[0].tolist()
    
    out = mg.querymany(ens_ids, scopes='ensembl.gene', fields='symbol', species='human')
    
    mapping = {}
    for item in out:
        if item and 'symbol' in item:
            mapping[item['query']] = item['symbol']
    
    # feature_name aktualisieren
    new_symbols = []
    for name in adata.var['feature_name']:
        if str(name).startswith('ENSG'):
            ens_id = str(name).split('.')[0]
            new_symbols.append(mapping.get(ens_id, name))
        else:
            new_symbols.append(name)
    
    adata.var['feature_name'] = new_symbols
    print(f"Mapped {len(mapping)} unique ENSEMBL IDs")

# var_names setzen
adata.var_names = adata.var['feature_name']

# Duplikate entfernen
n_dup = adata.var_names.duplicated().sum()
if n_dup > 0:
    print(f"Removing {n_dup} duplicate gene symbols (keeping first)...")
    adata = adata[:, ~adata.var_names.duplicated()].copy()

# Optional: Gene ohne Symbol entfernen
mask_na = adata.var_names.isna() | (adata.var_names == "")
if mask_na.any():
    print(f"Removing {mask_na.sum()} genes without symbol...")
    adata = adata[:, ~mask_na].copy()

print(f"Final: {adata.n_obs:,} cells, {adata.n_vars:,} genes")
print(f"Gene examples: {adata.var_names[:10].tolist()}")

# Speichern
print(f"Saving to {OUTPUT}...")
adata.write_h5ad(OUTPUT)
print("Done!")

Loading /net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/data/03_HIHAtlas/HumanImmuneHealthAtlas_all.h5ad...
Original: 1,821,725 cells, 32,357 genes
Using feature_name as gene symbols...
Mapping 7437 ENSEMBL-IDs to HGNC symbols...


18 input query terms found dup hits:	[('ENSG00000228044', 2), ('ENSG00000226506', 2), ('ENSG00000261600', 2), ('ENSG00000234162', 2), ('E
5 input query terms found no hit:	['ENSG00000272529', 'ENSG00000260206', 'ENSG00000266957', 'ENSG00000267058', 'ENSG00000269392']


Mapped 1338 unique ENSEMBL IDs
Removing 19 duplicate gene symbols (keeping first)...
Final: 1,821,725 cells, 32,338 genes
Gene examples: ['MIR1302-2HG', 'FAM138A', 'OR4F5', 'ENSG00000239945', 'ENSG00000239906', 'ENSG00000241599', 'ENSG00000236601', 'OR4F29', 'ENSG00000235146', 'OR4F16']
Saving to /net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/data/03_HIHAtlas/HIHA_hgnc.h5ad...
Done!


### Downsample Human Immune Health Atlas:
1. Remove all cells with disease == exclude_disease
2. For cell_types with < max_downsample: keep all
3. For cell_types with >= max_downsample: randomly sample max_downsample

In [4]:
import numpy as np

adata = sc.read_h5ad("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/data/03_HIHAtlas/HIHA_hgnc.h5ad")
checkpoint(f"Loaded: {adata.n_obs:,} cells, {adata.n_vars:,} genes")

BASE = "/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune"
OUTPUT_H5AD = f"{BASE}/data/03_HIHAtlas/HIHAtlas_downsampled.h5ad"

checkpoint("Starting downsampling pipeline")
def downsample_adata(adata, disease_col="disease", celltype_col="cell_type", 
                     exclude_disease="cytomegalovirus infection", min_keep_all=2000, max_downsample=5000):

    # 1. Remove disease
    checkpoint(f"Removing cells with {disease_col} == '{exclude_disease}'...")
    mask_disease = adata.obs[disease_col] != exclude_disease
    adata_filtered = adata[mask_disease].copy()
    checkpoint(f"After removing '{exclude_disease}': {adata_filtered.n_obs:,} cells (original: {adata.n_obs:,})")
    
    # 2. Downsampling per celltype
    checkpoint(f"Downsampling per {celltype_col} (keep all ≤{max_downsample}, downsample >{max_downsample} to {max_downsample})...")
    cells_to_keep = []
    
    for celltype in sorted(adata_filtered.obs[celltype_col].unique()):
        mask_ct = adata_filtered.obs[celltype_col] == celltype
        n_cells = mask_ct.sum()
        
        if n_cells <= max_downsample:
            # Keep all
            cells_to_keep.extend(adata_filtered.obs_names[mask_ct])
            checkpoint(f"  {celltype}: {n_cells:,} (keep all)")
        else:
            # Downsample
            sampled_idx = np.random.choice(
                adata_filtered.obs_names[mask_ct], 
                size=max_downsample, 
                replace=False
            )
            cells_to_keep.extend(sampled_idx)
            checkpoint(f"  {celltype}: {n_cells:,} -> {max_downsample:,} (downsampled)")
    
    adata_downsampled = adata_filtered[cells_to_keep].copy()
    checkpoint(f"Final: {adata_downsampled.n_obs:,} cells (reduced from {adata.n_obs:,})")
    
    return adata_downsampled


# Downsample
adata_downsampled = downsample_adata(adata)

# Save
checkpoint(f"Saving downsampled AnnData to {OUTPUT_H5AD}...")
adata_downsampled.write(OUTPUT_H5AD)
checkpoint(f"Saved: {OUTPUT_H5AD}")

checkpoint("Done!")


[2026-08-26 10:04:31] Loaded: 1,821,725 cells, 32,338 genes
[2026-08-26 10:04:31] Starting downsampling pipeline
[2026-08-26 10:04:31] Removing cells with disease == 'cytomegalovirus infection'...
[2026-08-26 10:04:41] After removing 'cytomegalovirus infection': 1,053,414 cells (original: 1,821,725)
[2026-08-26 10:04:41] Downsampling per cell_type (keep all ≤5000, downsample >5000 to 5000)...
[2026-08-26 10:04:41]   Axl+ dendritic cell, human: 296 (keep all)
[2026-08-26 10:04:41]   Be cell: 6,264 -> 5,000 (downsampled)
[2026-08-26 10:04:41]   CD14-low, CD16-positive monocyte: 26,860 -> 5,000 (downsampled)
[2026-08-26 10:04:41]   CD14-positive, CD16-negative classical monocyte: 156,628 -> 5,000 (downsampled)
[2026-08-26 10:04:41]   CD141-positive myeloid dendritic cell: 551 (keep all)
[2026-08-26 10:04:41]   CD16-negative, CD56-bright natural killer cell, human: 6,813 -> 5,000 (downsampled)
[2026-08-26 10:04:41]   CD16-positive, CD56-dim natural killer cell, human: 82,787 -> 5,000 (down

In [ ]:
hiha_annot = sc.read_h5ad("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/data/03_HIHAtlas/HIHAtlas_ds_TICAannot.h5ad")

print(hiha_annot.obs.columns)
hiha_annot.var.columns


Index(['cohort.cohortGuid', 'sample.sampleKitGuid', 'specimen.specimenGuid',
       'pipeline.fileGuid', 'subject.birthYear', 'subject.ageAtFirstDraw',
       'subject.ageGroup', 'subject.race', 'subject.ethnicity', 'subject.cmv',
       'subject.bmi', 'sample.visitName', 'sample.drawYear',
       'sample.subjectAgeAtDraw', 'batch_id', 'pool_id', 'chip_id', 'well_id',
       'original_barcodes', 'cell_name', 'n_reads', 'n_umis', 'n_genes',
       'total_counts_mito', 'pct_counts_mito', 'doublet_score', 'AIFI_L1',
       'AIFI_L2', 'AIFI_L3', 'disease_ontology_term_id',
       'tissue_ontology_term_id', 'tissue_type', 'suspension_type',
       'assay_ontology_term_id', 'sex_ontology_term_id', 'donor_id',
       'self_reported_ethnicity_ontology_term_id',
       'development_stage_ontology_term_id', 'cell_type_ontology_term_id',
       'is_primary_data', 'cell_type', 'assay', 'disease', 'sex', 'tissue',
       'self_reported_ethnicity', 'development_stage', 'observation_joinid',
       '

Index(['mito', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts',
       'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts',
       'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean',
       'std', 'gene_symbols', 'feature_is_filtered', 'feature_name',
       'feature_reference', 'feature_biotype', 'feature_length',
       'feature_type'],
      dtype='object')

In [33]:
X = hiha_annot.X
print(X[:5, :5])          # rohe Werte, kleine Stichprobe
print(X.max())       
print(hiha_annot.uns.get('log1p', 'kein log1p Eintrag gefunden'))
print(hiha_annot.layers.keys())  # gibt es evtl. eine 'counts' oder 'lognorm' Layer?  

print('-----')
vals = hiha_annot.X.data
print("min:", vals.min(), "max:", vals.max())
print("median:", np.median(vals))
print("99.9 percentile:", np.percentile(vals, 99.9))

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 0 stored elements and shape (5, 5)>
106.46361
kein log1p Eintrag gefunden
KeysView(Layers with keys: )
-----
min: -3.1186411 max: 106.46361
median: -0.18354462
99.9 percentile: 15.98226


In [36]:
print(hiha_annot.raw)
if hiha_annot.raw is not None:
    print(hiha_annot.raw.X[:5,:5])
    print(hiha_annot.raw.X.max())

Raw AnnData with n_obs × n_vars = 129435 × 32357
    var: 'mito', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'gene_symbols', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 0 stored elements and shape (5, 5)>
24099.0


In [37]:
vals = hiha_annot.raw.X.data
print("min:", vals.min(), "max:", vals.max())
print("Sind alle Werte ganzzahlig?", np.allclose(vals, np.round(vals)))

min: 1.0 max: 24099.0
Sind alle Werte ganzzahlig? True
